# Paper — 03: Mask Corrections vs. Gradient Estimator

**Produces:** `figures_paper/fig3_corrections_fail.pdf`, `figures_paper/fig4_gradient_works.pdf`

Built on `exp_orientation_6_to_9.ipynb`. Adds Gaussian blur and Shapely smoothing as
corrections 1–2, then runs all 6 mask-based approaches plus Exp 7 (Canny gradient).

**Corrections tested (all fail):**
1. Gaussian blur on binary mask
2. Shapely polygon smoothing (buffer trick)
3. Exp 6 — SAM2 logit isocontour
4. Exp 8 — Soft-mask image moments
5. Exp 9 — Point-prompt reprompting
6. Baseline SAM2 (box-prompted) — included as the reference pipeline being corrected

**Exp 7 (Canny gradient) — works.**

**Requires GPU** (`--gres=gpu:1` on Sherlock).

In [ ]:
import sys
sys.path.insert(0, "/scratch/users/cayleigh/YOLOv8-BeyondEarth/src")

import typing_extensions
if not hasattr(typing_extensions, "TypeIs"):
    typing_extensions.TypeIs = typing_extensions.TypeGuard

import torch
import torch._utils
if not hasattr(torch, "_utils"):
    torch._utils = sys.modules["torch._utils"]

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import geopandas as gpd
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from shapely.geometry import Polygon
from shapely import segmentize
from skimage.measure import find_contours
from scipy.stats import kstest
import warnings

from sahi import AutoDetectionModel
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

from rastertools_BOULDERING import convert as raster_convert, metadata as raster_metadata
from shptools_BOULDERING.geometry import fitEllipse
from shptools_BOULDERING.geomorph import boulder_row

In [ ]:
# ── GPU check ────────────────────────────────────────────────────────────
torch.cuda.empty_cache()
if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA device.\n"
        "  On Sherlock: srun --gres=gpu:1 --mem=32G --pty bash\n"
        "  Then: Kernel → Restart Kernel and re-run")
print(f"GPU: {torch.cuda.get_device_name(0)}")

# ── Paths ─────────────────────────────────────────────────────────────────
prieur_dir      = Path("/scratch/users/cayleigh/Apr2023-Mars-Moon-Earth-mask-5px")
prieur_test_dir = prieur_dir / "preprocessing" / "test"
work_dir        = Path.home() / "tmp" / "YOLOv8BeyondEarth"
in_raster       = Path("/scratch/users/cayleigh/test_raster/M1221383405.tif")
tile_tmp_dir    = work_dir / "tile_tmp"; tile_tmp_dir.mkdir(parents=True, exist_ok=True)

gt_tile_ids = ["1386", "1503", "2054", "2277", "2508"]

model_weights   = work_dir / "yolov8_model" / "yolov8-m-boulder-detection-tmp.pt"
sam2_checkpoint = Path("/scratch/users/cayleigh/checkpoints/sam2.1_hiera_small.pt")

# ── Filters & style ───────────────────────────────────────────────────────
tile_res             = raster_metadata.get_resolution(in_raster)[0]
AREAL_THRESHOLD      = (tile_res ** 2) * (4.74 ** 2)
AR_MIN, AR_MAX       = 1.2, 2.0
CONFIDENCE_THRESHOLD = 0.10
MIN_AREA_THRESHOLD   = 6

OUT_DIR = Path("figures_paper"); OUT_DIR.mkdir(exist_ok=True)
BINS    = np.linspace(0, 180, 37)
plt.rcParams.update({"font.size": 9, "axes.titlesize": 9, "figure.dpi": 150})
print(f"Tile resolution: {tile_res:.4f} m/px")

In [ ]:
torch.cuda.empty_cache()

detection_model = AutoDetectionModel.from_pretrained(
    model_type="yolov8",
    model_path=model_weights.as_posix(),
    confidence_threshold=CONFIDENCE_THRESHOLD,
    device="cuda:0",
    image_size=1024)

sam2_base_model = build_sam2(
    "configs/sam2.1/sam2.1_hiera_s.yaml", sam2_checkpoint, device="cuda:0")
predictor = SAM2ImagePredictor(sam2_base_model)

print("Models loaded.")

## Data collection

Same as `exp_orientation_6_to_9.ipynb` — YOLO + SAM2 on the 5 test tiles.
Each record stores: `box_mask`, `box_logit`, `point_mask`, `tile_image`, `bbox`, `poly_area`.

In [ ]:
def load_tile_rgb(tile_tif, tile_tmp):
    tile_png = tile_tmp / f"{tile_tif.stem}.png"
    if not tile_png.exists():
        raster_convert.tiff_to_png(tile_tif, tile_png)
    return np.array(Image.open(tile_png).convert("RGB"))


def collect_tile_detections(tile_image_np):
    """
    YOLO detection + box-prompted SAM2 + point-prompted SAM2 on a single tile.
    Returns list of per-detection dicts.
    predict_batch returns (masks, iou_scores, low_res_logits) — we keep the logits for Exp 6/8.
    """
    with torch.no_grad():
        yolo_results = detection_model.model(
            [tile_image_np], imgsz=detection_model.image_size,
            verbose=False, device=detection_model.device)

    boxes_data = yolo_results[0].boxes.data
    conf_mask  = boxes_data[:, 4] >= CONFIDENCE_THRESHOLD
    boxes_data = boxes_data[conf_mask]
    if len(boxes_data) == 0:
        return []

    bboxes = boxes_data[:, :4].cpu().numpy()
    scores = boxes_data[:, 4].cpu().numpy()

    # Box-prompted SAM2 — captures logit maps for Exp 6 and 8
    with torch.no_grad():
        predictor.set_image_batch([tile_image_np])
        box_masks_batch, _, box_logits_batch = predictor.predict_batch(
            None, None, box_batch=[bboxes], multimask_output=False)
    box_masks  = box_masks_batch[0][:, 0]   # (N, H, W) bool
    box_logits = box_logits_batch[0][:, 0]  # (N, 256, 256) float

    # Point-prompted SAM2 — centroid of box_mask as prompt (Exp 9)
    H, W = tile_image_np.shape[:2]
    point_masks = np.zeros((len(bboxes), H, W), dtype=bool)
    with torch.no_grad():
        predictor.set_image(tile_image_np)
        for i in range(len(bboxes)):
            yx = np.argwhere(box_masks[i])
            if len(yx):
                cx, cy = float(yx[:, 1].mean()), float(yx[:, 0].mean())
            else:
                cx = (bboxes[i, 0] + bboxes[i, 2]) / 2
                cy = (bboxes[i, 1] + bboxes[i, 3]) / 2
            pm, _, _ = predictor.predict(
                point_coords=np.array([[cx, cy]]),
                point_labels=np.array([1]),
                multimask_output=False)
            point_masks[i] = pm[0]

    records = []
    for i in range(len(bboxes)):
        area = int(box_masks[i].sum())
        if area <= MIN_AREA_THRESHOLD:
            continue
        records.append({
            "bbox":       bboxes[i],
            "score":      float(scores[i]),
            "box_mask":   box_masks[i],
            "box_logit":  box_logits[i],
            "point_mask": point_masks[i],
            "tile_image": tile_image_np,
            "poly_area":  area * (tile_res ** 2),
        })
    return records

In [ ]:
all_records = []
for tile_id in gt_tile_ids:
    tile_tif = prieur_test_dir / "images" / f"M1221383405_{tile_id}_image.tif"
    tile_np  = load_tile_rgb(tile_tif, tile_tmp_dir)
    recs     = collect_tile_detections(tile_np)
    for r in recs:
        r["tile_id"] = tile_id
    all_records.extend(recs)
    print(f"  tile {tile_id}: {len(recs)} detections")

print(f"\nTotal: {len(all_records)} detections across {len(gt_tile_ids)} tiles")

## Orientation estimators

Each function takes a detection record and returns `(angle180, aspect_ratio)` or `None`.

- **Baseline + corrections 1–2** (Gaussian blur, Shapely): mask-based, no SAM2 required
- **Exp 6, 8** (logit isocontour, soft moments): use `box_logit` from SAM2 predict_batch
- **Exp 9** (point-prompt): uses `point_mask` collected above
- **Exp 7** (Canny gradient): reads from raw image — no mask shape used

In [ ]:
def poly_to_orient(poly):
    """segmentize → fitEllipse → boulder_row → (angle180, aspect_ratio)."""
    row_seg = pd.Series({"geometry": segmentize(poly, tile_res)})
    ellipse_poly, _, _, _ = fitEllipse(row_seg)
    mrr_row = pd.Series({"geometry": ellipse_poly.minimum_rotated_rectangle})
    vals = boulder_row(mrr_row)
    long_ax, short_ax, angle180 = vals[2], vals[3], vals[7]
    ar = long_ax / short_ax if short_ax > 0 else None
    return angle180, ar


def binary_mask_to_poly(mask_bool):
    mask_u8   = mask_bool.astype(np.uint8) * 255
    cnts, _   = cv2.findContours(mask_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not cnts:
        return None
    pts = max(cnts, key=cv2.contourArea).squeeze(1)
    return Polygon(pts.astype(float)) if len(pts) >= 4 else None


# ── Baseline: binary mask → polygon → full pipeline ───────────────────────
def orient_baseline(rec):
    poly = binary_mask_to_poly(rec["box_mask"])
    if poly is None:
        return None
    try:
        return poly_to_orient(poly)
    except Exception:
        return None


# ── Correction 1: Gaussian blur ───────────────────────────────────────────
# Soften staircase edges then rethreshold — the pixel grid is reintroduced at threshold
def orient_gauss_blur(rec, sigma=2.0, thresh=0.5):
    blurred = cv2.GaussianBlur(rec["box_mask"].astype(np.float32), (0, 0), sigma)
    mask_u8 = (blurred >= thresh).astype(np.uint8) * 255
    cnts, _ = cv2.findContours(mask_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not cnts:
        return None
    pts  = max(cnts, key=cv2.contourArea).squeeze(1)
    poly = Polygon(pts.astype(float)) if len(pts) >= 4 else None
    if poly is None:
        return None
    try:
        return poly_to_orient(poly)
    except Exception:
        return None


# ── Correction 2: Shapely polygon smoothing ───────────────────────────────
# Buffer outward then inward to round corners — vertices still anchor to pixel grid
def orient_shapely_smooth(rec, buffer_px=2.0, simplify_tol=1.0):
    poly = binary_mask_to_poly(rec["box_mask"])
    if poly is None or not poly.is_valid:
        return None
    smoothed = poly.buffer(buffer_px).buffer(-buffer_px).simplify(simplify_tol)
    if smoothed.is_empty:
        return None
    try:
        return poly_to_orient(smoothed)
    except Exception:
        return None


# ── Exp 6: SAM2 logit isocontour ──────────────────────────────────────────
# Extract the 0-level contour of the raw logit map — sub-pixel in the logit space,
# but the logit map is on a 256×256 grid, so staircase structure is still present
def orient_exp6(rec):
    logit_2d    = rec["box_logit"].astype(float)
    contours_rc = find_contours(logit_2d, level=0.0)
    if not contours_rc:
        return None
    contour_rc = max(contours_rc, key=len)
    if len(contour_rc) < 5:
        return None
    poly = Polygon(contour_rc[:, ::-1])
    if not poly.is_valid or poly.is_empty:
        return None
    try:
        return poly_to_orient(poly)
    except Exception:
        return None


# ── Exp 8: Soft-mask image moments ────────────────────────────────────────
# Orientation from logit-weighted covariance — uses full 256×256 logit map
# (cropping by bbox caused coord-space mismatch; full map is already localized)
def orient_exp8(rec):
    logit_2d = rec["box_logit"].astype(float)
    weights  = np.maximum(logit_2d, 0.0)
    total    = weights.sum()
    if total < 1e-6:
        return None
    rows, cols = np.indices(weights.shape)
    cx = (weights * cols).sum() / total
    cy = (weights * rows).sum() / total
    dx, dy = cols - cx, rows - cy
    mu20 = (weights * dx**2).sum() / total
    mu02 = (weights * dy**2).sum() / total
    mu11 = (weights * dx * dy).sum() / total
    theta_rad = 0.5 * np.arctan2(2.0 * mu11, mu20 - mu02)
    angle180  = (90.0 - np.degrees(theta_rad)) % 180
    trace = mu20 + mu02
    det   = mu20 * mu02 - mu11**2
    disc  = max(0.0, (trace / 2)**2 - det)
    lam_max = trace / 2 + np.sqrt(disc)
    lam_min = trace / 2 - np.sqrt(disc)
    ar = np.sqrt(lam_max / lam_min) if lam_min > 1e-9 else None
    return angle180, ar


# ── Exp 9: Point-prompted SAM2 ────────────────────────────────────────────
# Point prompt (centroid) instead of box prompt — mask is still pixel-aligned binary
def orient_exp9(rec):
    poly = binary_mask_to_poly(rec["point_mask"])
    if poly is None or int(rec["point_mask"].sum()) <= MIN_AREA_THRESHOLD:
        return None
    try:
        return poly_to_orient(poly)
    except Exception:
        return None


# ── Exp 7: Canny edge → cv2.fitEllipse ───────────────────────────────────
# Reads orientation from real image edges — bypasses mask shape entirely.
# Run Canny on the FULL unmasked tile first (not the masked image), then
# restrict to the dilated mask region. This avoids the artificial boundary
# edge that masking-before-Canny would introduce.
def orient_exp7(rec):
    tile_gray    = cv2.cvtColor(rec["tile_image"], cv2.COLOR_RGB2GRAY)
    mask_u8      = rec["box_mask"].astype(np.uint8)
    edges_full   = cv2.Canny(tile_gray, threshold1=15, threshold2=45)
    dilated_mask = cv2.dilate(mask_u8, np.ones((3, 3), np.uint8), iterations=2)
    edges        = cv2.bitwise_and(edges_full, edges_full, mask=dilated_mask)
    edge_pts     = np.argwhere(edges)
    if len(edge_pts) < 5:
        return None
    edge_xy = edge_pts[:, ::-1].astype(np.float32).reshape(-1, 1, 2)
    try:
        (_, _), (MA, ma), angle_deg = cv2.fitEllipse(edge_xy)
    except cv2.error:
        return None
    if ma <= 0:
        return None
    ar       = max(MA, ma) / min(MA, ma)
    angle180 = angle_deg % 180
    return angle180, ar


print("Estimator functions defined.")

In [ ]:
# Load GT orientations — same pipeline as baseline, for overlay on histograms
gt_records = []
for tile_id in gt_tile_ids:
    gt_shp = prieur_test_dir / "labels" / f"M1221383405_{tile_id}_mask.shp"
    if not gt_shp.exists():
        continue
    gdf_gt = gpd.read_file(gt_shp)
    for geom in tqdm(gdf_gt.geometry, desc=f"GT tile {tile_id}", leave=False):
        if geom is None or geom.is_empty:
            continue
        try:
            row_seg = pd.Series({"geometry": segmentize(geom, tile_res)})
            ellipse_poly, _, _, _ = fitEllipse(row_seg)
            mrr_row = pd.Series({"geometry": ellipse_poly.minimum_rotated_rectangle})
            vals = boulder_row(mrr_row)
            long_ax, short_ax, angle180 = vals[2], vals[3], vals[7]
            ar = long_ax / short_ax if short_ax > 0 else None
        except Exception:
            angle180, ar = None, None
        gt_records.append({"angle180": angle180, "aspect_ratio": ar})

df_gt       = pd.DataFrame(gt_records)
df_gt_elong = df_gt[(df_gt["aspect_ratio"] >= AR_MIN) & (df_gt["aspect_ratio"] <= AR_MAX)]
gt_angles   = df_gt_elong["angle180"].dropna().values
print(f"GT elongated boulders (AR {AR_MIN}–{AR_MAX}): {len(gt_angles)}")

In [ ]:
ESTIMATORS = [
    ("Baseline (binary mask)",  orient_baseline),
    ("Gaussian blur",           orient_gauss_blur),
    ("Shapely smoothing",       orient_shapely_smooth),
    ("Exp 6 (logit isocontour)", orient_exp6),
    ("Exp 8 (soft moments)",    orient_exp8),
    ("Exp 9 (point-prompt)",    orient_exp9),
    ("Exp 7 (Canny gradient)",  orient_exp7),
]

orient_results = {}

for name, fn in ESTIMATORS:
    angles = []
    for rec in tqdm(all_records, desc=name):
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            try:
                result = fn(rec)
            except Exception:
                result = None
        if result and result[1] is not None and AR_MIN <= result[1] <= AR_MAX:
            angles.append(result[0])
    orient_results[name] = np.array(angles)
    print(f"  {name}: {len(angles)} elongated boulders")

## Figures

In [ ]:
def hist_ax(angles, ax, title, color, gt_angles=None):
    counts, _ = np.histogram(angles, bins=BINS)
    cx = (BINS[:-1] + BINS[1:]) / 2
    ax.bar(cx, counts, width=4.5, color=color, edgecolor="white", lw=0.3)
    if gt_angles is not None:
        gc, _ = np.histogram(gt_angles, bins=BINS)
        ax.step(cx, gc * len(angles) / max(len(gt_angles), 1),
                where="mid", color="seagreen", lw=1.2, label="GT (scaled)")
        ax.legend(fontsize=7)
    ax.axhline(len(angles) / len(cx), color="k", ls="--", lw=0.6, alpha=0.5)
    ax.set(xlim=(0, 180), xticks=[0, 45, 90, 135, 180],
           xlabel="angle180 (°)", ylabel="Count")
    ax.set_title(f"{title}\n(n={len(angles)})")
    ax.spines[["top", "right"]].set_visible(False)

In [ ]:
# Figure 3 — all mask-based corrections fail
# Includes baseline + the 5 corrections (excludes Exp 7 which works)
correction_names = [n for n in orient_results if n != "Exp 7 (Canny gradient)"]
n_cols = len(correction_names)

fig, axes = plt.subplots(1, n_cols, figsize=(2.8 * n_cols, 2.8))
COLORS = ["tomato"] + ["#4C72B0"] * (n_cols - 1)   # baseline red, corrections blue

for ax, name, color in zip(axes, correction_names, COLORS):
    hist_ax(orient_results[name], ax, name, color, gt_angles=gt_angles)

fig.tight_layout()
fig.savefig(OUT_DIR / "fig3_corrections_fail.pdf", bbox_inches="tight")
plt.show()
print("Saved fig3_corrections_fail.pdf")

In [ ]:
# Figure 4 — gradient estimator works
fig, axes = plt.subplots(1, 3, figsize=(9, 2.8))

hist_ax(gt_angles,
        axes[0], "GT (Prieur et al.)",         "seagreen")
hist_ax(orient_results["Baseline (binary mask)"],
        axes[1], "Baseline (binary mask)",     "tomato",   gt_angles=gt_angles)
hist_ax(orient_results["Exp 7 (Canny gradient)"],
        axes[2], "Exp 7 (Canny gradient)",     "#DD8452",  gt_angles=gt_angles)

fig.tight_layout()
fig.savefig(OUT_DIR / "fig4_gradient_works.pdf", bbox_inches="tight")
plt.show()
print("Saved fig4_gradient_works.pdf")

In [ ]:
# Summary statistics — KS test vs. uniform and 90° spike ratio
expected_per_bin = None

print(f"{'Method':<35}  {'n':>6}  {'KS-D':>6}  {'p-value':>10}  {'90° spike ratio':>16}")
print("-" * 82)

all_names_angles = [("GT (Prieur et al.)", gt_angles)] + list(orient_results.items())

for name, angles in all_names_angles:
    n = len(angles)
    if n == 0:
        print(f"{name:<35}  {'0':>6}  {'—':>6}  {'—':>10}  {'—':>16}")
        continue
    D, p = kstest(angles / 180.0, "uniform")
    # Spike ratio: count in [85°, 95°] relative to expected uniform count
    spike_90 = ((angles >= 85) & (angles <= 95)).sum()
    expected = n / 36   # expected per 5° bin if uniform
    ratio    = spike_90 / max(expected, 1e-9)
    print(f"{name:<35}  {n:>6}  {D:>6.3f}  {p:>10.2e}  {ratio:>16.2f}x")

print()
print("90° spike ratio ≈ 1.0 → flat (correct);  >> 1 → spike present")